# 05 - BFS（Breadth-First Search，廣度優先搜尋）

## 學習目標

完成本 Notebook 後，你將能夠：

- 說明 BFS「一層一層向外擴展」的搜尋方式。
- 使用 `collections.deque` 實作佇列。
- 使用 BFS 走訪圖。
- 在無權圖中找出邊數最少的路徑。
- 分析 BFS 的時間與空間複雜度。


## 1. 範例圖

本 Notebook 使用以下無向圖：

```text
A-B, A-C, B-D, B-E, C-F, E-F
```

圖以鄰接串列表示，每個鍵是節點，值是可以直接到達的鄰居。


In [ ]:
graph = {
    "A": ["B", "C"],
    "B": ["A", "D", "E"],
    "C": ["A", "F"],
    "D": ["B"],
    "E": ["B", "F"],
    "F": ["C", "E"],
}

for node, neighbors in graph.items():
    print(f"{node} 的鄰居：{neighbors}")


## 2. BFS 的核心想法

BFS 從起點開始，按照距離逐層處理：

1. 先拜訪距離起點 0 條邊的起點。
2. 再拜訪距離起點 1 條邊的所有節點。
3. 接著拜訪距離起點 2 條邊的所有節點。
4. 持續進行，直到佇列為空。

BFS 使用佇列（Queue），遵守 FIFO（First In, First Out，先進先出）：

- `append()` 從右側加入。
- `popleft()` 從左側取出最早加入的節點。

Python 應使用 `collections.deque`；不要使用清單的 `pop(0)`，因為每次都要搬動其餘元素，效率較差。


In [ ]:
from collections import deque


def bfs(graph: dict[str, list[str]], start: str) -> list[str]:
    '''使用 BFS 回傳從 start 開始的拜訪順序。'''
    if start not in graph:
        return []

    # 起點加入佇列時就標記，避免它被其他節點重複加入。
    visited = {start}
    queue = deque([start])
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)

        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return order


order = bfs(graph, "A")
print("BFS 拜訪順序：", order)
assert order == ["A", "B", "C", "D", "E", "F"]


## 3. 逐步觀察佇列

每次取出節點後，印出佇列剩下的內容，可以看到 BFS 如何先完成同一層，再進入下一層。


In [ ]:
def trace_bfs(graph: dict[str, list[str]], start: str) -> None:
    visited = {start}
    queue = deque([start])
    step = 0

    while queue:
        step += 1
        node = queue.popleft()
        print(f"第 {step} 步：取出 {node}，目前佇列 {list(queue)}")

        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
                print(f"  發現 {neighbor}，加入後為 {list(queue)}")


trace_bfs(graph, "A")


## 4. 無權圖的最短路徑

BFS 按照距離逐層搜尋，所以第一次到達某個節點時，使用的邊數一定最少。

為了重建路徑，使用 `parent` 記錄每個節點是從哪個節點被發現的。例如 `parent["D"] = "B"` 代表搜尋時由 `B` 到達 `D`。

找到目標後，從目標沿著 `parent` 倒著走回起點，再反轉清單即可。


In [ ]:
def shortest_path_bfs(
    graph: dict[str, list[str]], start: str, goal: str
) -> list[str] | None:
    '''回傳無權圖中從 start 到 goal 的最短路徑；無法到達則回傳 None。'''
    if start not in graph or goal not in graph:
        return None

    queue = deque([start])
    parent = {start: None}

    while queue:
        node = queue.popleft()

        if node == goal:
            break

        for neighbor in graph.get(node, []):
            # 出現在 parent 中就代表已經發現過。
            if neighbor not in parent:
                parent[neighbor] = node
                queue.append(neighbor)

    if goal not in parent:
        return None

    # 從 goal 沿著 parent 回到 start。
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()
    return path


path = shortest_path_bfs(graph, "A", "F")
print("A 到 F 的最短路徑：", path)
assert path == ["A", "C", "F"]
assert shortest_path_bfs(graph, "D", "C") == ["D", "B", "A", "C"]


## 5. 距離表

如果只想知道起點到每個節點的最少邊數，可以在發現鄰居時設定：

```python
distance[neighbor] = distance[node] + 1
```


In [ ]:
def bfs_distances(
    graph: dict[str, list[str]], start: str
) -> dict[str, int]:
    '''回傳起點到每個可到達節點的最少邊數。'''
    if start not in graph:
        return {}

    distance = {start: 0}
    queue = deque([start])

    while queue:
        node = queue.popleft()
        for neighbor in graph.get(node, []):
            if neighbor not in distance:
                distance[neighbor] = distance[node] + 1
                queue.append(neighbor)

    return distance


distances = bfs_distances(graph, "A")
print("從 A 出發的距離：", distances)
assert distances == {"A": 0, "B": 1, "C": 1, "D": 2, "E": 2, "F": 2}


## 6. DFS 與 BFS 比較

| 比較項目 | DFS | BFS |
|---|---|---|
| 主要資料結構 | 遞迴堆疊或 `stack` | `queue` |
| 搜尋方式 | 一路往深處走，再回溯 | 按距離逐層擴展 |
| 無權圖最短路徑 | 不保證 | 保證第一次到達即為最短 |
| 常見用途 | 回溯、連通元件、偵測環 | 最短邊數、層級走訪、擴散問題 |

兩者使用鄰接串列時，時間複雜度都是 $O(V+E)$；最壞情況空間複雜度都是 $O(V)$。


## 7. 常見錯誤與練習

常見錯誤：

- 使用 `list.pop(0)` 代替 `deque.popleft()`。
- 等到節點出列才標記，讓同一節點被重複加入佇列。
- 忘記處理起點或終點不存在的情況。
- 在有權重的圖中直接使用 BFS 求最短距離；普通 BFS 只適用於每條邊代價相同的情況。

**練習：**實作 `nodes_at_distance(graph, start, k)`，回傳與 `start` 距離恰好為 `k` 的所有節點。


In [ ]:
def nodes_at_distance(
    graph: dict[str, list[str]], start: str, k: int
) -> list[str]:
    # TODO：先呼叫或參考 bfs_distances，再挑出距離等於 k 的節點。
    pass


# 完成後可取消註解測試：
# assert nodes_at_distance(graph, "A", 2) == ["D", "E", "F"]
